In [1]:
from IPython.display import Image

In [3]:
Image(url='https://storage.googleapis.com/gweb-uniblog-publish-prod/images/agentic-vision-gemini-3_flash_bl.width-1000.format-webp_z5u5YjZ.webp', width=800)

- agentic vision
    - https://blog.google/innovation-and-ai/technology/developers-tools/agentic-vision-gemini-3-flash/
    - tools
        - bbox, crop, rotate, draw, annotate, save
        - crop & zoom 也是一种稳定注意力，实现模型视觉能力的聚焦
- 对比之前的一些 visual promting 的方案
    - 对原始图片进行一些必要的处理，形成一个 visual scratchpad 对于 agentic agent 而言，是自主产生的，而非显式地外部 pipeline 预定义的；
    - 释放模型的基础视觉能力，提升整体的表现
        - 基础视觉能力：检测/识别，open-world general knowledge
- online demos
    - https://aistudio.google.com/apps/bundled/gemini_visual_thinking?e=0&showPreview=true&showAssistant=true&fullscreenApplet=true
        - https://arxiv.org/pdf/2412.18613
        - 似乎这些 illusion（视错觉） 都可以通过某种 detection & crop or annotate/label count 的方式解决掉
        - maybe trigger prompts
            - Crop out all
            - Zoom in to see 、Zoom and rotate crop
            - annotate them on the image
- research for AI Agent
    - novelty：某些特定 topic or tasks，之前不可行，现在变得可行，之前有一定效果，现在效果有更大提升；
    - agentic：基于能力提升 + tool（code）+ multiturn

### gemini api tools

- https://ai.google.dev/gemini-api/docs/tools?hl=zh-cn
    - https://ai.google.dev/gemini-api/docs/code-execution?hl=zh-cn

### examples

```python
from google import genai
from google.genai import types
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(), override=True)

client = genai.Client()

image = types.Part.from_uri(
    file_uri="https://goo.gle/instrument-img",
    mime_type="image/jpeg",
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[image, "Zoom into the expression pedals and tell me how many pedals are there?"],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    ),
)

print(response.text)
```

- `response.executable_code`
    - 会去 crop 图像，crop 的区域是模型生成的，因为这个代码就是模型生成的，代码中会包含
- `response.code_execution_result`
- 交互过程，也可以在 ai studio 中尝试；

### 抓包分析

在客户端上，tools 并没有被序列化成一个“文本 Sequence”，而是止步于 JSON 对象。
真正的“Sequence 形成”（即把工具定义变成模型能读懂的 Prompt 或 Token 流）完全是在 Google 的服务端 发生的黑盒操作。

```python
from google import genai
from google.genai import types
from dotenv import load_dotenv, find_dotenv

assert load_dotenv(find_dotenv(), override=True)

client = genai.Client()

# 抓包分析：添加 request 和 response 的 hook
def log_request(request):
    print(f"\n=== Request ===\n{request.method} {request.url}")
    try:
        print(f"Body:\n{request.content.decode('utf-8')}")
    except Exception:
        print(f"Body: <binary> ({len(request.content)} bytes)")

def log_response(response):
    print(f"\n=== Response ===\n{response.status_code}")
    try:
        response.read()
        print(f"Body:\n{response.content.decode('utf-8')}")
    except Exception:
        print(f"Body: <binary> ({len(response.content)} bytes)")

# 注入 hook 到 client.models 使用的 httpx client
client.models._api_client._httpx_client.event_hooks['request'].append(log_request)
client.models._api_client._httpx_client.event_hooks['response'].append(log_response)

image = types.Part.from_uri(
    file_uri="https://goo.gle/instrument-img",
    mime_type="image/jpeg",
)

response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=[image, "Zoom into the expression pedals and tell me how many pedals are there?"],
    config=types.GenerateContentConfig(
        tools=[types.Tool(code_execution=types.ToolCodeExecution)]
    ),
)

print(response.text)
```

- 完整的 Prompt (Request Body)

```json
{
  "contents": [
    {
      "parts": [
        {
          "fileData": {
            "fileUri": "https://goo.gle/instrument-img",
            "mimeType": "image/jpeg"
          }
        },
        {
          "text": "Zoom into the expression pedals and tell me how many pedals are there?"
        }
      ],
      "role": "user"
    }
  ],
  "tools": [
    {
      "codeExecution": {}
    }
  ],
  "generationConfig": {}
}
```

- 完整的 Response (Response Body)
    - 响应中包含了模型生成的 Python 代码 (executableCode) 以及执行结果 (codeExecutionResult)，最后才是基于执行结果生成的文本回复。
    - parts：多轮工具调用
 
```json
{
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "executableCode": {
              "language": "PYTHON",
              "code": "import PIL.Image\nimport PIL.ImageDraw\n\n# Load the image..."
            }
          },
          {
            "codeExecutionResult": {
              "outcome": "OUTCOME_OK",
              "output": "Original size: 2048x1362\n"
            }
          },
          {
            "text": "Based on the zoomed-in image, there are **4** expression pedals..."
          }
        ],
        "role": "model"
      },
      "finishReason": "STOP",
      "index": 0
    }
  ],
  "usageMetadata": { ... }
}
```

### multi-turn

```json
{
  "request": {
    "url": "https://generativelanguage.googleapis.com/v1beta/models/gemini-3-flash-preview:generateContent",
    "method": "POST",
    "body": {
      "contents": [
        {
          "parts": [
            {
              "inlineData": {
                "data": "_9j_4AAQSkZJRgABAQAAAQABAAD...(省略)",
                "mimeType": "image/jpeg"
              }
            },
            {
              "text": "Crop out all the animals, and use them as icons in a matplotlib plot showing the lifespan of those animals. Sort by lifespan."
            }
          ],
          "role": "user"
        }
      ],
      "tools": [
        {
          "codeExecution": {}
        }
      ],
      "generationConfig": {}
    }
  },
  
  "response": {
    "statusCode": 200,
    "body": {
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "executableCode": {
                  "language": "PYTHON",
                  "code": "import cv2\nimport numpy as np\nimport matplotlib.pyplot as plt...(检测和裁剪动物的代码)"
                },
                "thoughtSignature": "EpEeCo4eAXLI2nyFZsJFCw17g...(省略)"
              },
              {
                "codeExecutionResult": {
                  "outcome": "OUTCOME_OK",
                  "output": "Detected 39 animals.\n"
                }
              },
              {
                "executableCode": {
                  "language": "PYTHON",
                  "code": "from matplotlib.offsetbox import OffsetImage...(创建图表的代码)"
                },
                "thoughtSignature": "EqU1CqI1AXLI2nzEfF7RsAQw8h...(省略)"
              },
              {
                "inlineData": {
                  "mimeType": "image/png",
                  "data": "iVBORw0KGgoAAAANSUhEUgAABdwAAAM...(省略)"
                }
              },
              {
                "inlineData": {
                  "mimeType": "image/png",
                  "data": "iVBORw0KGgoAAAANSUhEUgAABdwAAAM...(省略)"
                }
              },
              {
                "text": "\nThe animals have been cropped from the original image and sorted by their approximate lifespans. The resulting bar chart uses the animals' own images as icons to illustrate their typical lifespans in years."
              }
            ],
            "role": "model"
          },
          "finishReason": "STOP",
          "index": 0
        }
      ],
      
      "usageMetadata": {
        "promptTokenCount": 4774,
        "candidatesTokenCount": 41,
        "totalTokenCount": 24663,
        "promptTokensDetails": [
          {
            "modality": "IMAGE",
            "tokenCount": 3252
          },
          {
            "modality": "TEXT",
            "tokenCount": 1522
          }
        ],
        "toolUsePromptTokenCount": 17027,
        "toolUsePromptTokensDetails": [
          {
            "modality": "IMAGE",
            "tokenCount": 4344
          },
          {
            "modality": "TEXT",
            "tokenCount": 12683
          }
        ],
        "thoughtsTokenCount": 2821
      },
      
      "modelVersion": "gemini-3-flash-preview",
      "responseId": "Y3N9aYnlCcib1e8P-beVoQo"
    }
  },
  
  "summary": {
    "task": "图像处理 + 数据可视化",
    "inputImage": {
      "format": "jpeg",
      "sizeTokens": 3252
    },
    "outputImages": [
      {
        "format": "png",
        "description": "第一张处理结果图"
      },
      {
        "format": "png", 
        "description": "第二张处理结果图"
      }
    ],
    "codeExecutionSteps": 2,
    "animalsDetected": 39,
    "thoughtsTokenCount": 2821,
    "totalTokens": 24663
  }
}
```